# Data layer exploration

Inspects the Sprint 0 data pipeline against the committed synthetic sample.

**This notebook implements nothing.** Every transformation below calls the same
importable functions in `src/squadopt/` that the test suite exercises. If a cell
here disagrees with the package, the package is right.

No plots: charting would add a dependency the project does not need for Sprint 0,
so the exploration is deliberately table-based. Outputs are not committed — run the
cells to populate them.

In [ ]:
from pathlib import Path

import pandas as pd

from squadopt.data import KEY_COLUMNS, SourceAdapter, build_canonical_dataset, load_csv
from squadopt.features import build_feature_dataset, feature_column_names
from squadopt.features.config import DEFAULT_FEATURE_CONFIG
from squadopt.prediction import build_projection_table

pd.set_option("display.width", 120)


def repository_root() -> Path:
    """Walk upwards to the directory holding pyproject.toml."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the repository.")


SAMPLE_FILE = repository_root() / "data" / "sample" / "raw_player_gameweeks.csv"
SEASON = "2025-26"

# Mirrors the tested definition in tests/fixtures/synthetic_gameweeks.py. Declared
# inline so the notebook needs no sys.path manipulation to import the test package.
SAMPLE_ADAPTER = SourceAdapter(
    name="synthetic-sample",
    column_map={
        "season_label": "season",
        "gw": "gameweek",
        "player_ref": "player_id",
        "display_name": "name",
        "club_ref": "team_id",
        "pos_code": "position",
        "price": "price_tenths",
        "minutes_played": "minutes",
        "points": "total_points",
    },
    position_codes={"1": "GK", "2": "DEF", "3": "MID", "4": "FWD"},
    price_unit="units",
)
SAMPLE_FILE

## 1. What the source actually looks like

Read as text on purpose. Type inference is a silent transformation: it strips
leading zeros from identifiers and promotes an integer column to float as soon as
one row is blank, which is exactly what makes the optimizer reject a price column.

In [ ]:
raw = load_csv(SAMPLE_FILE)
print(raw.shape)
print(raw.dtypes.to_string())
raw.head()

## 2. The canonical dataset

Adapted, cleaned, validated, and deterministically ordered. Note the dtypes:
quantities and identifiers are non-nullable `int64`, because the optimizer checks
`price_tenths` element-wise against `numbers.Integral`.

In [ ]:
canonical = build_canonical_dataset(raw, adapter=SAMPLE_ADAPTER)
print(canonical.dtypes.to_string())
canonical.head()

## 3. Shape and quality of the panel

Before trusting any feature, confirm the grid is complete, the key is unique, and
the distributions are not degenerate.

In [ ]:
players = canonical["player_id"].nunique()
gameweeks = canonical["gameweek"].nunique()
print(f"players={players}  gameweeks={gameweeks}  rows={len(canonical)}")
print(f"complete grid: {len(canonical) == players * gameweeks}")
print(f"duplicate keys: {int(canonical.duplicated(subset=list(KEY_COLUMNS)).sum())}")
print(f"missing values: {int(canonical.isna().sum().sum())}")
print()
print(canonical["position"].value_counts().to_string())
print()
canonical[["price_tenths", "minutes", "total_points"]].describe()

In [ ]:
# Two facts that matter downstream: some gameweeks are blank (zero minutes), and
# realized points can be negative. Neither is an error, and neither is clamped.
print(f"blank gameweeks: {int((canonical['minutes'] == 0).sum())}")
print(f"negative scores: {int((canonical['total_points'] < 0).sum())}")
print()
canonical.groupby("position")["price_tenths"].agg(["min", "median", "max"])

## 4. Features, and why gameweek 1 is empty

Each feature for gameweek `t` is grouped by `(season, player_id)` and shifted one
gameweek before its window is applied. Read one player's rows across: the value on
row `t` is built from the rows *above* it, never from row `t` itself.

In [ ]:
features = build_feature_dataset(canonical)
FEATURES = list(feature_column_names(DEFAULT_FEATURE_CONFIG))

one_player = features.loc[features["player_id"] == features["player_id"].min()]
one_player[["gameweek", "minutes", "total_points", *FEATURES]]

In [ ]:
# Missingness is concentrated entirely in gameweek 1, where no history exists.
# It is never back-filled: importing a later value would import the future.
features.groupby("gameweek")[FEATURES].apply(lambda block: block.isna().sum())

## 5. Checking the leakage claim by hand

The test suite proves this five ways; this cell lets you watch it. Delete every
gameweek from 5 onwards and recompute. If features for gameweeks 1-4 are identical,
then nothing about them depended on the future.

In [ ]:
CUT = 5


def early_features(frame: pd.DataFrame) -> pd.DataFrame:
    built = build_feature_dataset(frame)
    early = built.loc[built["gameweek"] < CUT]
    return early[["player_id", "gameweek", *FEATURES]].reset_index(drop=True)


baseline = early_features(canonical)

truncated = canonical.loc[canonical["gameweek"] < CUT].reset_index(drop=True)
deleted_ok = early_features(truncated).equals(baseline)

mutated = canonical.copy(deep=True)
mutated.loc[mutated["gameweek"] >= CUT, "total_points"] = 999
rewritten_ok = early_features(mutated).equals(baseline)

print(f"unchanged when gameweeks >= {CUT} never existed: {deleted_ok}")
print(f"unchanged when future results are rewritten: {rewritten_ok}")

## 6. The optimizer-ready table

Exactly six columns for one target gameweek. Identity, club, position, and price
come from the target row itself, because all four are fixed at that gameweek's
deadline; `expected_points` comes only from shifted features.

In [ ]:
projections = build_projection_table(features, season=SEASON, gameweek=6)
print(list(projections.columns))
print(projections["expected_points"].describe().to_string())
projections.sort_values("expected_points", ascending=False).head(10)

In [ ]:
# Gameweek 1 has no history at all, so every projection falls back to the declared
# per-position constant. A rolling-only baseline genuinely cannot rank players yet;
# see docs/data_followups.md for the price-based prior proposed for Sprint 1.
opening = build_projection_table(features, season=SEASON, gameweek=1)
print(f"distinct projections in gameweek 1: {opening['expected_points'].nunique()}")
opening.head()

## Where to go next

- [Data contract](../docs/data_contract.md) — schemas and time-of-knowledge rules
- [Data dictionary](../docs/data_dictionary.md) — per-column meaning and leakage risk
- [Data pipeline](../docs/data_pipeline.md) — stage responsibilities and leakage controls
- [Follow-up work](../docs/data_followups.md) — deliberate gaps and how to close them

For the full chain including the solver, run `python -m scripts.run_pipeline_demo`.